In [9]:
import numpy as np

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectFromModel
from sklearn.linear_model import LogisticRegression

In [5]:
# load dataset
data = load_breast_cancer()

X = data.data      # features
y = data.target    # labels
print (data.target)
print("Shape of X:", X.shape)
print("Shape of y:", y.shape)


[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 1 0 0 0 0 0 0 0 0 1 0 1 1 1 1 1 0 0 1 0 0 1 1 1 1 0 1 0 0 1 1 1 1 0 1 0 0
 1 0 1 0 0 1 1 1 0 0 1 0 0 0 1 1 1 0 1 1 0 0 1 1 1 0 0 1 1 1 1 0 1 1 0 1 1
 1 1 1 1 1 1 0 0 0 1 0 0 1 1 1 0 0 1 0 1 0 0 1 0 0 1 1 0 1 1 0 1 1 1 1 0 1
 1 1 1 1 1 1 1 1 0 1 1 1 1 0 0 1 0 1 1 0 0 1 1 0 0 1 1 1 1 0 1 1 0 0 0 1 0
 1 0 1 1 1 0 1 1 0 0 1 0 0 0 0 1 0 0 0 1 0 1 0 1 1 0 1 0 0 0 0 1 1 0 0 1 1
 1 0 1 1 1 1 1 0 0 1 1 0 1 1 0 0 1 0 1 1 1 1 0 1 1 1 1 1 0 1 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 1 1 1 1 1 1 0 1 0 1 1 0 1 1 0 1 0 0 1 1 1 1 1 1 1 1 1 1 1 1
 1 0 1 1 0 1 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 0 1 1 1 0 1 0 1 1 1 1 0 0 0 1 1
 1 1 0 1 0 1 0 1 1 1 0 1 1 1 1 1 1 1 0 0 0 1 1 1 1 1 1 1 1 1 1 1 0 0 1 0 0
 0 1 0 0 1 1 1 1 1 0 1 1 1 1 1 0 1 1 1 0 1 1 0 0 1 1 1 1 1 1 0 1 1 1 1 1 1
 1 0 1 1 1 1 1 0 1 1 0 1 1 1 1 1 1 1 1 1 1 1 1 0 1 0 0 1 0 1 1 1 1 1 0 1 1
 0 1 0 1 1 0 1 0 1 1 1 1 1 1 1 1 0 0 1 1 1 1 1 1 0 1 1 1 1 1 1 1 1 1 1 0 1
 1 1 1 1 1 1 0 1 0 1 1 0 

In [6]:
# create classifier
model = LogisticRegression(max_iter=5000)


In [7]:
scores = cross_val_score(
    model,
    X,
    y,
    cv=10,
    scoring="accuracy"
)

print("Accuracy for each fold:", scores)
print("Mean accuracy:", scores.mean())
print("Standard deviation:", scores.std())

Accuracy for each fold: [0.98245614 0.9122807  0.92982456 0.94736842 0.98245614 0.98245614
 0.92982456 0.94736842 0.96491228 0.96428571]
Mean accuracy: 0.9543233082706767
Standard deviation: 0.023770661464399972


In [10]:
baseline = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=5000))
])

cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

baseline_scores = cross_val_score(
    baseline, X, y, cv=cv, scoring="accuracy"
)

print("Baseline accuracy mean:", baseline_scores.mean())
print("Baseline accuracy std:", baseline_scores.std())


Baseline accuracy mean: 0.9754385964912279
Baseline accuracy std: 0.01953601530817553


In [11]:
# L1 model for feature selection
l1_lr = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("l1", LogisticRegression(
        penalty="l1",
        solver="liblinear",
        C=1.0,
        max_iter=5000
    ))
])

l1_lr.fit(X, y)

# coefficients live inside the final step
coef = l1_lr.named_steps["l1"].coef_

# coef shape is (n_classes, n_features), here binary so (1, n_features)
nonzero = np.sum(coef != 0)
total = coef.size

print("Total coefficients:", total)
print("Non-zero coefficients:", nonzero)
print("Selected features (non-zero):", nonzero, "out of", X.shape[1])


Total coefficients: 30
Non-zero coefficients: 18
Selected features (non-zero): 18 out of 30


In [12]:
fs5_pipeline = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("select", SelectFromModel(
        LogisticRegression(
            penalty="l1",
            solver="liblinear",
            C=1.0,
            max_iter=5000
        )
    )),
    ("clf", LogisticRegression(max_iter=5000))
])

fs5_scores = cross_val_score(
    fs5_pipeline, X, y, cv=cv, scoring="accuracy"
)

print("FS5 accuracy mean:", fs5_scores.mean())
print("FS5 accuracy std:", fs5_scores.std())


FS5 accuracy mean: 0.9719298245614034
FS5 accuracy std: 0.019536015308175506


In [13]:
print("Baseline mean accuracy:", baseline_scores.mean())
print("FS5 mean accuracy:", fs5_scores.mean())
print("Difference (FS5 - Baseline):", fs5_scores.mean() - baseline_scores.mean())


Baseline mean accuracy: 0.9754385964912279
FS5 mean accuracy: 0.9719298245614034
Difference (FS5 - Baseline): -0.0035087719298244613
